# Model 3: Damage Level Classification

**Goal:** Given a cropped image of a building, classify damage severity: None, Minor, Major, Destroyed.

Since xBD assesses damage on a *per-building* basis, the data loading step involves parsing the JSON polygons, isolating the building from the high-res PNG, and making patches.

In [ ]:
import os
import json
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

## 1. Class and Loss Definition
Building Damage is an ordinal classification problem (No damage < Minor < Major < Destroyed). We use Categorical Crossentropy, but weighted loss or Focal Loss is better for imbalanced disaster datasets.

In [ ]:
# Class Mapping
damage_map = {
    "no-damage": 0,
    "minor-damage": 1,
    "major-damage": 2,
    "destroyed": 3
}
num_classes = 4

# Advanced Model replacing basic CNN
patch_input = layers.Input(shape=(128, 128, 3)) # Patches of buildings

# Preprocessing layers attached to network directly
x = layers.Rescaling(1./255)(patch_input)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)

# Feature Extractor
x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs=patch_input, outputs=outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # To improve further, implement Focal Loss for Imbalanced learning
    metrics=['accuracy']
)

model.summary()